# Phase 6.0-B: High-Affinity Regulatory Network Analysis

**Project**: Human LncRNA Atlas

**Objective**: Identify core lncRNAs with strong regulatory capabilities

**Data Source**: `/api/v1/export/high-affinity` API

## 1. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import requests
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']

# Species name mapping (Chinese to English)
SPECIES_MAP = {'人类': 'Human', '黑猩猩': 'Chimpanzee', '猕猴': 'Macaque', '狨猴': 'Marmoset'}

API_BASE_URL = 'http://localhost:8000/api/v1'
MIN_BA = 100
LIMIT = 10000

print(f'Analysis started: {datetime.now()}')
print(f'Parameters: MIN_BA={MIN_BA}, LIMIT={LIMIT}')

In [ ]:
# Fetch data from API
response = requests.get(f'{API_BASE_URL}/export/high-affinity',
    params={'min_ba': MIN_BA, 'limit': LIMIT, 'format': 'json'})

if response.status_code == 200:
    data = response.json()
    df = pd.DataFrame(data['data'])
    # Convert Chinese species names to English
    df['species_name'] = df['species_name'].map(SPECIES_MAP)
    print(f'Fetched {len(df)} records')
else:
    print(f'API failed: {response.status_code}')

df.head()

## 2. Data Quality Check

In [ ]:
print('Data Quality Report')
print(f'Dimensions: {df.shape}')
print('Species distribution:')
print(df['species_name'].value_counts())

df_clean = df.drop_duplicates()
print(f'After dedup: {len(df_clean)} records')
df_clean.to_csv('data/high_affinity_raw_data.csv', index=False)

## 3. Binding Affinity Analysis

In [ ]:
print('Binding Affinity Statistics:')
print(df_clean['binding_affinity'].describe())
print(f'Median: {df_clean["binding_affinity"].median():.2f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(df_clean['binding_affinity'], bins=50, edgecolor='black', alpha=0.7)
axes[0,0].set_xlabel('Binding Affinity')
axes[0,0].set_ylabel('Frequency')
axes[0,0].set_title('BA Distribution', fontweight='bold')

df_clean.boxplot(column='binding_affinity', by='species_name', ax=axes[0,1])
axes[0,1].set_title('BA by Species', fontweight='bold')
plt.suptitle('')

for sp in df_clean['species_name'].unique():
    axes[1,0].hist(df_clean[df_clean['species_name']==sp]['binding_affinity'], bins=30, alpha=0.5, label=sp, density=True)
axes[1,0].set_title('BA Density by Species', fontweight='bold')
axes[1,0].legend()

for sp in df_clean['species_name'].unique():
    d = df_clean[df_clean['species_name']==sp]['binding_affinity'].sort_values()
    axes[1,1].plot(d, np.arange(1,len(d)+1)/len(d), label=sp, lw=2)
axes[1,1].set_title('CDF', fontweight='bold')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('figures/01_ba_distribution_analysis.png', dpi=300)
print('Figure saved')
plt.show()

## 4. Top 100 lncRNA Ranking

In [ ]:
lncrna_stats = df_clean.groupby(['lncrna_gene_id','lncrna_name','species_name']).agg(
    {'target_gene_id':'count', 'binding_affinity':['mean','max','std']}).reset_index()
lncrna_stats.columns = ['lncrna_gene_id','lncrna_name','species_name','target_count','avg_ba','max_ba','std_ba']
lncrna_stats['composite_score'] = lncrna_stats['target_count'] * lncrna_stats['avg_ba']

top_100 = lncrna_stats.nlargest(100, 'composite_score')
top_100['rank'] = range(1, 101)

print('Top 20 High-Affinity lncRNAs:')
print(top_100.head(20))
top_100.to_excel('results/top_100_high_affinity_lncrnas.xlsx', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
top20 = top_100.head(20)
axes[0].barh(top20['lncrna_name'], top20['target_count'], color='steelblue')
axes[0].set_xlabel('Target Count')
axes[0].invert_yaxis()
axes[0].set_title('Top 20 lncRNAs', fontweight='bold')

sc = axes[1].scatter(top_100['target_count'], top_100['avg_ba'], c=top_100['composite_score'], s=100, cmap='viridis', alpha=0.6)
axes[1].set_xlabel('Target Count')
axes[1].set_ylabel('Avg BA')
axes[1].set_title('lncRNA Characteristics', fontweight='bold')
plt.colorbar(sc, ax=axes[1], label='Composite Score')

plt.tight_layout()
plt.savefig('figures/02_top_lncrnas_visualization.png', dpi=300)
plt.show()

## 5. Network Construction

In [ ]:
G = nx.DiGraph()
for _, r in df_clean.iterrows():
    G.add_node(r['lncrna_name'], node_type='lncRNA', species=r['species_name'])
    G.add_node(r['target_name'], node_type='target_gene')
    G.add_edge(r['lncrna_name'], r['target_name'], weight=r['binding_affinity'])

print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Density: {nx.density(G):.6f}')

## 6. Centrality Analysis

In [ ]:
degree_cent = nx.degree_centrality(G)
between_cent = nx.betweenness_centrality(G, weight='weight')

cent_df = pd.DataFrame([{'lncrna': n, 'degree': degree_cent[n], 'betweenness': between_cent[n]}
    for n,d in G.nodes(data=True) if d.get('node_type')=='lncRNA'])
cent_df = cent_df.sort_values('betweenness', ascending=False)

print('Top 15 by Betweenness:')
print(cent_df.head(15))
cent_df.to_excel('results/lncrna_centrality_analysis.xlsx', index=False)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].hist(cent_df['degree'], bins=30, edgecolor='black')
axes[0,0].set_title('Degree Centrality', fontweight='bold')
axes[1,0].hist(cent_df['betweenness'], bins=30, color='lightgreen', edgecolor='black')
axes[1,0].set_title('Betweenness Centrality', fontweight='bold')

top15 = cent_df.head(15).set_index('lncrna')[['degree','betweenness']]
top15_norm = (top15 - top15.min()) / (top15.max() - top15.min())
sns.heatmap(top15_norm.T, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1,1])
axes[1,1].set_title('Top 15 Multi-Metric', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/03_centrality_analysis.png', dpi=300)
plt.show()

## 7. Community Detection

In [ ]:
try:
    import community as community_louvain
    partition = community_louvain.best_partition(G.to_undirected(), weight='weight')
    modularity = community_louvain.modularity(partition, G.to_undirected())
    print(f'Communities: {len(set(partition.values()))}, Modularity: {modularity:.4f}')
except ImportError:
    print('python-louvain not installed')
    partition = None

## 8. Network Visualization

In [ ]:
from matplotlib.patches import Patch

top30 = top_100.head(30)['lncrna_name'].tolist()
subnodes = set(top30)
for lnc in top30:
    if lnc in G:
        subnodes.update(G.successors(lnc))
G_sub = G.subgraph(subnodes).copy()

plt.figure(figsize=(18, 14))
pos = nx.spring_layout(G_sub, k=2, seed=42)
colors = ['salmon' if G_sub.nodes[n].get('node_type')=='lncRNA' else 'lightblue' for n in G_sub.nodes()]
sizes = [300 + 3000*degree_cent.get(n,0) for n in G_sub.nodes()]

nx.draw_networkx_nodes(G_sub, pos, node_color=colors, node_size=sizes, alpha=0.8)
nx.draw_networkx_edges(G_sub, pos, alpha=0.3, arrows=True)
nx.draw_networkx_labels(G_sub, pos, font_size=6)

plt.legend(handles=[Patch(facecolor='salmon', label='lncRNA'), Patch(facecolor='lightblue', label='Target')], loc='upper right')
plt.title('Regulatory Network (Top 30)', fontweight='bold')
plt.axis('off')
plt.savefig('figures/04_regulatory_network_visualization.png', dpi=300)
plt.show()

## 9. Summary

In [ ]:
print('='*60)
print('KEY FINDINGS')
print('='*60)
print(f'Records: {len(df_clean)} (BA > {MIN_BA})')
print(f'lncRNAs: {len(cent_df)}')
print(f'Top lncRNA: {top_100.iloc[0]["lncrna_name"]} ({top_100.iloc[0]["species_name"]})')
print(f'Network density: {nx.density(G):.6f}')
print(f'Key hub: {cent_df.iloc[0]["lncrna"]} (betweenness={cent_df.iloc[0]["betweenness"]:.4f})')
print('='*60)

## 10. Export Data

In [ ]:
nodes_df = pd.DataFrame([{'node_id':n, 'type':d.get('node_type'), 'degree':degree_cent.get(n,0)} for n,d in G.nodes(data=True)])
nodes_df.to_csv('results/network_nodes.csv', index=False)
edges_df = pd.DataFrame([{'source':u, 'target':v, 'weight':d['weight']} for u,v,d in G.edges(data=True)])
edges_df.to_csv('results/network_edges.csv', index=False)
print('Network data exported to results/')